# 3. Portfolio evaluation and scientific discussion

Load the saved predictions and frozen selection. This notebook does not train models or choose a new competition entry. Report the assignment and competition experiments separately.

In [ ]:
from pathlib import Path
import os
import sys

# Launch from the repository root. All handoffs are explicit disk artifacts.
PROJECT = Path.cwd()
if not (PROJECT / 'stockml').is_dir():
    raise RuntimeError('Open this notebook with the repository root as the working directory.')
sys.path.insert(0, str(PROJECT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT / '.cache' / 'matplotlib'))
import pandas as pd
from IPython.display import display, Markdown
from stockml.common import root, smoke, load_json

pd.set_option('display.max_columns', 20)
print('Artifacts:', root())
print('SYNTHETIC SMOKE TEST — NOT FINANCIAL RESULTS' if smoke() else 'REAL-DATA EXPERIMENT')

In [ ]:
from stockml.models import check_frozen
frozen = check_frozen()
display(pd.Series(frozen['selected_entry']))
display(pd.read_csv(root() / 'reports' / 'validation_portfolios.csv'))

## Allocation, timing, costs, and accounting

| Strategy | Rule |
|---|---|
| Equal weight | One-eighth in each stock; rebalance every five sessions |
| Return-ranked ML | One-third in each of the three highest predicted-return stocks |
| Risk-aware ML | Exclude risk probability ≥0.5, rank the rest, and fill up to three one-third slots |
| SPY | Buy at the first execution open and hold |

Missing risk-aware slots remain in cash. Ties use alphabetical ticker order. Both algorithms and both pooled/individual layouts are reported. Only the entry selected by validation net Sharpe (then lower turnover, then strategy name) is the competition entry.

Each strategy starts with $100,000, uses fractional adjusted units, no leverage, no cash interest, and 10 basis points per dollar bought or sold. Signals from a completed close execute at the next session's open. Rebalancing uses drifted holdings and solves for post-cost wealth before allocating, so costs cannot create borrowing.

Track close-to-close portfolio values daily, including the initial allocation cost and first open-to-close return. Annualize with 252 sessions; use a zero risk-free rate for Sharpe. Drawdown includes the initial $100,000 peak. Terminal wealth is marked to market without liquidation. Turnover is total absolute dollars traded divided by pre-trade wealth at each rebalance; it is a two-sided, cumulative measure.

The required five-day return target is close-to-close; executable portfolio returns start at the following open. They are deliberately not treated as identical. Final portfolio valuation may include sessions whose five-day predictive targets are still unknown.

In [ ]:
from stockml.reporting import report_portfolios, evidence_discussion
assignment_performance, assignment_daily, assignment_trades = report_portfolios('assignment')

In [ ]:
evidence_discussion('assignment')

## Independent competition results

The following chart and table report the frozen entry alongside all predeclared comparators. Do not replace the entry with whichever comparator wins on 2026. A partial-year result is labeled year-to-date and includes its final completed session.

In [ ]:
competition_performance, competition_daily, competition_trades = report_portfolios('competition')

In [ ]:
evidence_discussion('competition')

## Interpretation checklist and limitations

Use the generated evidence above and Notebook 2's importance/ablation tables to defend these conclusions:

1. Which algorithm wins on return RMSE and risk ROC-AUC? Are the winners consistent across stocks?
2. Which task shows more reliable improvement over its own baseline? Do not compare RMSE units with AUC units.
3. Does pooling help each algorithm, given that individual models reuse pooled tuning settings?
4. Does the **frozen** entry outperform SPY after costs, and what happens to drawdown and volatility?
5. Do better predictions correspond to better ranked portfolios? Use the joined prediction/investment table.
6. Does the evidence support generalization across both periods? A short holdout and overlapping labels do not establish statistical significance.
7. Which financial features matter on validation? Compare top-feature identities and the feature ablation before asserting that additional features helped.

The model universe was selected retrospectively and is concentrated in large surviving companies. Regime shifts, Yahoo data revisions, simplified fills, fixed costs, and zero cash yield constrain interpretation. A longer preregistered forward test, point-in-time universe, block-bootstrap uncertainty intervals, and richer execution assumptions would strengthen a future experiment. Do not tune these changes on 2026 after seeing its results.

A successful project can conclude that the models did not beat simple baselines.

## AI assistance and independent verification

AI assistance was used to structure the experiment, draft the shared Python implementation, and create these notebooks and test cases. AI-generated code and conclusions require independent review.

The verification suite checks future-price perturbations, five-day targets against hand calculations, split purging, feature-order rejection, saved-model prediction parity, trade timing, and portfolio accounting against hand-computed constant-price examples. In particular, the transaction-cost implementation uses post-cost wealth rather than allocating all pre-cost cash and implicitly borrowing the fees.

Before submission, run `python -m pytest -q`, inspect these calculations yourself, and add a sentence describing a suggestion or implementation **you personally verified or corrected**. Do not claim personal verification solely because automated tests pass.